# Agent Data Quality Audit

Comprehensive evaluation of data coverage and quality metrics for each agent's input data.

**CRITICAL:** Team Analyst degree coverage must be >= 40% or signal is too weak.

## Imports & Setup

In [149]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## Load Data

In [150]:
companies = pd.read_parquet("../data/processed/companies_clean.parquet")
print(f"Filtered dataset: {len(companies)} companies")
print(f"Date range: {companies['founded_at'].dt.year.min()}-{companies['founded_at'].dt.year.max()}")

# Split raw object tables (aligned with data_processing notebook)
companies_raw = pd.read_csv("../data/raw/companies.csv", low_memory=False)
financial_orgs = pd.read_csv("../data/raw/financial_orgs.csv", low_memory=False)
products = pd.read_csv("../data/raw/products.csv", low_memory=False)

# Other raw tables used in audit joins
rounds = pd.read_csv("../data/raw/funding_rounds.csv", low_memory=False)
milestones = pd.read_csv("../data/raw/milestones.csv", low_memory=False)
relationships = pd.read_csv("../data/raw/relationships.csv", low_memory=False)
degrees = pd.read_csv("../data/raw/degrees.csv", low_memory=False)
people = pd.read_csv("../data/raw/people.csv", low_memory=False)

company_ids = set(companies['object_id'])

print(f"\nRaw data sizes:")
print(f"  companies_raw: {len(companies_raw)}")
print(f"  financial_orgs: {len(financial_orgs)}")
print(f"  products: {len(products)}")
print(f"  people: {len(people)}")
print(f"  funding_rounds: {len(rounds)}")
print(f"  milestones: {len(milestones)}")
print(f"  relationships: {len(relationships)}")
print(f"  degrees: {len(degrees)}")

Filtered dataset: 2653 companies
Date range: 2005-2013

Raw data sizes:
  companies_raw: 196553
  financial_orgs: 11652
  products: 27738
  people: 226709
  funding_rounds: 52928
  milestones: 39456
  relationships: 402878
  degrees: 109610


## 1. Market Analyst Inputs

In [151]:
cat_missing = companies['category_code'].isna().sum()
cat_missing_pct = 100 * cat_missing / len(companies)
cat_nunique = companies['category_code'].nunique()
cat_top = companies['category_code'].value_counts().head(15)

print(f"Category Code:")
print(f"  Missing: {cat_missing_pct:.2f}% ({cat_missing} rows)")
print(f"  Unique values: {cat_nunique}")
print(f"\n  Top 15:")
for i, (cat, count) in enumerate(cat_top.items(), 1):
    pct = 100 * count / len(companies)
    print(f"    {i:2d}. {cat:20s} {count:4d} ({pct:5.2f}%)")

print(f"\n  Sample companies by category:")
for cat in cat_top.head(3).index:
    sample = companies[companies['category_code'] == cat][['name', 'category_code', 'country_code']].head(2)
    print(f"\n    {cat}:")
    for _, row in sample.iterrows():
        print(f"      - {row['name'][:50]:50s} ({row['country_code']})")

Category Code:
  Missing: 1.96% (52 rows)
  Unique values: 38

  Top 15:
     1. web                   611 (23.03%)
     2. software              336 (12.66%)
     3. mobile                213 ( 8.03%)
     4. games_video           198 ( 7.46%)
     5. advertising           170 ( 6.41%)
     6. ecommerce             169 ( 6.37%)
     7. enterprise            148 ( 5.58%)
     8. other                  86 ( 3.24%)
     9. biotech                69 ( 2.60%)
    10. network_hosting        67 ( 2.53%)
    11. search                 53 ( 2.00%)
    12. public_relations       51 ( 1.92%)
    13. social                 46 ( 1.73%)
    14. hardware               40 ( 1.51%)
    15. cleantech              40 ( 1.51%)

  Sample companies by category:

    web:
      - FriendFeed                                         (USA)
      - Youlicit                                           (USA)

    software:
      - Galaxy                                             (USA)
      - PicMe                

In [152]:
country_missing = companies['country_code'].isna().sum()
country_missing_pct = 100 * country_missing / len(companies)
country_top = companies['country_code'].value_counts().head(20)

print(f"Country Code:")
print(f"  Missing: {country_missing_pct:.2f}% ({country_missing} rows)")
print(f"\n  Top 20:")
for i, (country, count) in enumerate(country_top.items(), 1):
    pct = 100 * count / len(companies)
    print(f"    {i:2d}. {country:6s} {count:4d} ({pct:5.2f}%)")

print(f"\n  Sample companies by country:")
for country in country_top.head(3).index:
    sample = companies[companies['country_code'] == country][['name', 'country_code', 'category_code']].head(2)
    print(f"\n    {country}:")
    for _, row in sample.iterrows():
        print(f"      - {row['name'][:45]:45s} ({row['category_code']})")

Country Code:
  Missing: 12.63% (335 rows)

  Top 20:
     1. USA    1658 (62.50%)
     2. GBR     160 ( 6.03%)
     3. CAN      88 ( 3.32%)
     4. DEU      45 ( 1.70%)
     5. ISR      40 ( 1.51%)
     6. IND      30 ( 1.13%)
     7. FRA      30 ( 1.13%)
     8. CHN      21 ( 0.79%)
     9. AUS      19 ( 0.72%)
    10. SGP      17 ( 0.64%)
    11. IRL      14 ( 0.53%)
    12. NLD      14 ( 0.53%)
    13. ESP      13 ( 0.49%)
    14. ITA      13 ( 0.49%)
    15. DNK      12 ( 0.45%)
    16. RUS      12 ( 0.45%)
    17. CHE       9 ( 0.34%)
    18. ARG       9 ( 0.34%)
    19. FIN       8 ( 0.30%)
    20. ZAF       8 ( 0.30%)

  Sample companies by country:

    USA:
      - FriendFeed                                    (web)
      - Mobclix                                       (mobile)

    GBR:
      - CityTherapy                                   (web)
      - Broad Street Digital                          (other)

    CAN:
      - Metranome                                     (mobi

In [153]:
print(f"\nMarket segmentation feasibility:")
if cat_nunique <= 20 and cat_missing_pct < 15:
    print(f"  YES - Can bin into ~10 buckets")
else:
    print(f"  NO - Need careful binning ({cat_nunique} categories, {cat_missing_pct:.1f}% missing)")


Market segmentation feasibility:
  NO - Need careful binning (38 categories, 2.0% missing)


## 2. Business Model Analyst Inputs

In [154]:
companies['overview_len'] = companies['overview'].str.len()
overview_stats = {
    'mean': companies['overview_len'].mean(),
    'median': companies['overview_len'].median(),
    'p25': companies['overview_len'].quantile(0.25),
    'p75': companies['overview_len'].quantile(0.75),
    'max': companies['overview_len'].max(),
}

print(f"Overview Length Distribution:")
print(f"  Mean:      {overview_stats['mean']:.0f} chars")
print(f"  Median:    {overview_stats['median']:.0f} chars")
print(f"  25th pct:  {overview_stats['p25']:.0f} chars")
print(f"  75th pct:  {overview_stats['p75']:.0f} chars")
print(f"  Max:       {overview_stats['max']:.0f} chars")

print(f"\n  Sample overviews (showing first 200 chars):")
for idx, (name, overview) in enumerate(companies[['name', 'overview']].head(3).values, 1):
    preview = overview[:200] + ('...' if len(overview) > 200 else '')
    print(f"\n    {idx}. {name}")
    print(f"       {preview}")

Overview Length Distribution:
  Mean:      783 chars
  Median:    637 chars
  25th pct:  459 chars
  75th pct:  934 chars
  Max:       12307 chars

  Sample overviews (showing first 200 chars):

    1. FriendFeed
       [FriendFeed](http://www.friendfeed.com) aims to be a one stop shop for all your social networking updates and news items.  The four founders were all team members at [Google](/company/google) and help...

    2. moviestring.com
       moviestring.com provides a place for user to create mini movie reviews, follow their friends reviews and create a list of the movies they love. with several ways to access your reviews, including web,...

    3. Mobclix
       Mobclix (www.mobclix.com) is the industry's largest mobile ad exchange network via its sophisticated open marketplace platform and comprehensive account management solution for iPhone application deve...


In [155]:
funding_zero_null = (companies['funding_total_usd'] == 0).sum() + companies['funding_total_usd'].isna().sum()
funding_zero_null_pct = 100 * funding_zero_null / len(companies)
companies_with_funding = companies[companies['funding_total_usd'] > 0].copy()
companies_with_funding['funding_log'] = np.log10(companies_with_funding['funding_total_usd'])
funding_p99 = companies['funding_total_usd'].quantile(0.99)

print(f"Funding (total_usd):")
print(f"  Zero or null: {funding_zero_null_pct:.2f}%")
print(f"  99th percentile: USD {funding_p99:,.0f}")
print(f"\n  Distribution (log10, for companies with funding > 0):")
for percentile in [10, 25, 50, 75, 90]:
    val = companies_with_funding['funding_log'].quantile(percentile / 100)
    usd_val = 10 ** val
    print(f"    {percentile:2d}th pct: 10^{val:.2f} = USD {usd_val:,.0f}")

print(f"\n  Sample funding data:")
sample_funding = companies[companies['funding_total_usd'] > 0][['name', 'funding_total_usd', 'funding_rounds']].nlargest(5, 'funding_total_usd')
for _, row in sample_funding.iterrows():
    print(f"    {row['name'][:40]:40s} USD {row['funding_total_usd']:>15,.0f} ({int(row['funding_rounds'])} rounds)")

Funding (total_usd):
  Zero or null: 40.41%
  99th percentile: USD 131,878,386

  Distribution (log10, for companies with funding > 0):
    10th pct: 10^4.78 = USD 60,000
    25th pct: 10^5.55 = USD 355,000
    50th pct: 10^6.30 = USD 2,000,000
    75th pct: 10^7.00 = USD 9,950,000
    90th pct: 10^7.48 = USD 30,500,000

  Sample funding data:
    Twitter                                  USD   1,160,166,511 (8 rounds)
    Groupon                                  USD   1,147,288,416 (7 rounds)
    Zynga                                    USD     860,213,000 (8 rounds)
    Better Place                             USD     835,750,000 (6 rounds)
    SolarCity                                USD     789,039,992 (13 rounds)


In [156]:
rounds_filtered = rounds[rounds['object_id'].isin(company_ids)].copy()
rounds_by_company = rounds_filtered.groupby('object_id').size()
rounds_by_type = rounds_filtered['funding_round_type'].value_counts()

print(f"Rounds (joined to filtered companies):")
print(f"  Total records: {len(rounds_filtered)}")
print(f"  Companies with rounds: {len(rounds_by_company)} ({100*len(rounds_by_company)/len(companies):.1f}%)")
print(f"  Avg per company: {rounds_by_company.mean():.2f}")
print(f"\n  Round type distribution:")
for round_type, count in rounds_by_type.head(9).items():
    pct = 100 * count / len(rounds_filtered)
    print(f"    {round_type:20s} {count:5d} ({pct:5.2f}%)")

print(f"\n  Sample round records:")
sample_rounds = rounds_filtered[['object_id', 'funding_round_type', 'raised_amount_usd', 'funded_at']].head(5)
for _, row in sample_rounds.iterrows():
    print(f"    {row['object_id'][:20]:20s} {row['funding_round_type']:15s} USD {row['raised_amount_usd']:>12,.0f} ({row['funded_at']})")

Rounds (joined to filtered companies):
  Total records: 3307
  Companies with rounds: 1840 (69.4%)
  Avg per company: 1.80

  Round type distribution:
    angel                 1113 (33.66%)
    series-a               815 (24.64%)
    venture                513 (15.51%)
    series-b               387 (11.70%)
    series-c+              251 ( 7.59%)
    other                  183 ( 5.53%)
    private-equity          31 ( 0.94%)
    post-ipo                13 ( 0.39%)
    crowdfunding             1 ( 0.03%)

  Sample round records:
    c:9                  series-a        USD    1,500,000 (2007-01-01)
    c:9                  series-b        USD   10,000,000 (2007-03-01)
    c:26                 series-a        USD   13,200,000 (2005-06-01)
    c:23                 series-a        USD   45,000,000 (2007-05-09)
    c:30                 series-a        USD   12,500,000 (2007-06-01)


## 3. Feasibility Analyst Inputs

In [157]:
milestones_filtered = milestones[milestones['object_id'].isin(company_ids)].copy()
companies_with_milestones = milestones_filtered['object_id'].nunique()
companies_with_milestones_pct = 100 * companies_with_milestones / len(companies)

print(f"Milestones Coverage:")
print(f"  Companies with >= 1: {companies_with_milestones} ({companies_with_milestones_pct:.2f}%)")

if len(milestones_filtered) > 0:
    print(f"\n  Sample milestone records:")
    sample_miles = milestones_filtered[['object_id', 'description', 'milestone_at']].head(5)
    for _, row in sample_miles.iterrows():
        desc = str(row['description'])[:50] if pd.notna(row['description']) else 'N/A'
        print(f"    {row['object_id'][:20]:20s} {desc:50s} ({row['milestone_at']})")

Milestones Coverage:
  Companies with >= 1: 629 (23.71%)

  Sample milestone records:
    c:12                 Survives iPhone 3G Stevenote                       (2008-06-09)
    c:3138               Twhirl announces support for Seesmic video playbac (2008-06-17)
    c:314                Reddit goes Open Source                            (2008-06-18)
    c:314                Adds the ability to create your own Reddits        (2008-01-22)
    c:1388               AOL Acquires Huffington Post                       (2011-02-06)


In [158]:
companies_with_2plus_rounds = (rounds_by_company >= 2).sum()
companies_with_2plus_rounds_pct = 100 * companies_with_2plus_rounds / len(companies)

print(f"Time-Between-Rounds Analysis:")
print(f"  Companies with >= 2 rounds: {companies_with_2plus_rounds} ({companies_with_2plus_rounds_pct:.2f}%)")

if companies_with_2plus_rounds > 0:
    def compute_time_intervals(group):
        if len(group) < 2:
            return []
        group_sorted = group.sort_values('funded_at')
        intervals = []
        for i in range(1, len(group_sorted)):
            dt = pd.to_datetime(group_sorted.iloc[i]['funded_at']) - pd.to_datetime(group_sorted.iloc[i-1]['funded_at'])
            if pd.notna(dt):
                intervals.append(dt.total_seconds() / (365.25 * 24 * 3600))
        return intervals

    all_intervals = []
    companies_2plus = rounds_by_company[rounds_by_company >= 2].index.tolist()
    for cid in companies_2plus:
        cr = rounds_filtered[rounds_filtered['object_id'] == cid]
        intervals = compute_time_intervals(cr)
        all_intervals.extend(intervals)

    if all_intervals:
        print(f"  Median interval: {np.median(all_intervals):.2f} years")
        print(f"  Mean interval: {np.mean(all_intervals):.2f} years")
        print(f"\n  Sample company round sequences (first 3):")
        for company in companies_2plus[:3]:
            crounds = rounds_filtered[rounds_filtered['object_id'] == company].sort_values('funded_at')
            print(f"\n    {company}:")
            for _, row in crounds.head(4).iterrows():
                print(f"      {row['funding_round_type']:15s} USD {row['raised_amount_usd']:>12,.0f} ({row['funded_at']})")

Time-Between-Rounds Analysis:
  Companies with >= 2 rounds: 764 (28.80%)
  Median interval: 0.94 years
  Mean interval: 1.13 years

  Sample company round sequences (first 3):

    c:10054:
      series-a        USD    4,000,000 (2005-03-01)
      series-b        USD   17,000,000 (2005-12-06)
      series-c+       USD   22,000,000 (2006-10-04)
      series-c+       USD   26,000,000 (2008-08-26)

    c:101312:
      angel           USD            0 (2011-07-01)
      series-a        USD    1,500,000 (2011-10-17)
      series-a        USD    2,500,000 (2013-03-25)

    c:10158:
      series-a        USD    1,500,000 (2007-06-15)
      series-a        USD    2,000,000 (2008-08-26)
      private-equity  USD   11,786,415 (2013-01-29)


## 4. Team Analyst Inputs [MOST CRITICAL]

In [159]:
relationships_filtered = relationships[relationships['relationship_object_id'].isin(company_ids)].copy()
companies_with_relationships = relationships_filtered['relationship_object_id'].nunique()
companies_with_relationships_pct = 100 * companies_with_relationships / len(companies)

print(f"Relationships Coverage:")
print(f"  Companies with >= 1 person: {companies_with_relationships} ({companies_with_relationships_pct:.2f}%)")

if len(relationships_filtered) > 0:
    print(f"\n  Sample relationship records:")
    sample_rel = relationships_filtered[['relationship_object_id', 'person_object_id', 'title', 'start_at']].head(5)
    for _, row in sample_rel.iterrows():
        title = str(row['title'])[:40] if pd.notna(row['title']) else 'N/A'
        print(f"    {row['relationship_object_id'][:15]:15s} -> Person {row['person_object_id'][:10]:10s} | {title:40s}")

Relationships Coverage:
  Companies with >= 1 person: 2294 (86.47%)

  Sample relationship records:
    c:9             -> Person p:30       | Founder & Chairman                      
    c:9             -> Person p:31       | Board Member                            
    c:9             -> Person p:35       | Board Member                            
    c:9             -> Person p:26       | Board of Directors                      
    c:12            -> Person p:46       | CEO, Co-Founder                         


In [160]:
if len(relationships_filtered) > 0:
    title_counts = relationships_filtered['title'].value_counts().head(20)
    print(f"Top 20 titles in linked relationships:")
    for i, (title, count) in enumerate(title_counts.items(), 1):
        pct = 100 * count / len(relationships_filtered)
        print(f"    {i:2d}. {str(title)[:50]:50s} {count:5d} ({pct:5.2f}%)")
    
    print(f"\n  Sample relationship records (first 5)::")
    for _, row in relationships_filtered[['person_object_id', 'title', 'start_at']].head(5).iterrows():
        title = str(row['title'])[:50] if pd.notna(row['title']) else 'N/A'
        start = str(row['start_at'])[:10] if pd.notna(row['start_at']) else 'N/A'
        print(f"    Person {row['person_object_id'][:12]:12s} | {title:50s} | Started: {start}")
else:
    print("No relationships found.")

Top 20 titles in linked relationships:
     1. CEO                                                  720 ( 5.75%)
     2. Co-Founder                                           392 ( 3.13%)
     3. Board Member                                         384 ( 3.07%)
     4. CTO                                                  376 ( 3.00%)
     5. Founder                                              324 ( 2.59%)
     6. Advisor                                              299 ( 2.39%)
     7. Board of Directors                                   281 ( 2.24%)
     8. Director                                             227 ( 1.81%)
     9. Co-founder                                           180 ( 1.44%)
    10. CFO                                                  178 ( 1.42%)
    11. Investor                                             169 ( 1.35%)
    12. COO                                                  162 ( 1.29%)
    13. Founder & CEO                                        123 ( 0.98%)

In [161]:
if len(relationships_filtered) > 0:
    person_ids = set(relationships_filtered['person_object_id'].unique())
    people_filtered = people[people['object_id'].isin(person_ids)].copy()
    title_counts = relationships_filtered['title'].value_counts().head(20)
    print(f"Top 20 titles in linked people:")
    for i, (title, count) in enumerate(title_counts.items(), 1):
        pct = 100 * count / len(relationships_filtered)
        print(f"    {i:2d}. {str(title)[:50]:50s} {count:5d} ({pct:5.2f}%)")
    
    # Join relationships to people so first_name comes from people and title comes from relationships.
    people_name = people_filtered[['object_id', 'first_name']].drop_duplicates('object_id')
    rel_people_sample = relationships_filtered[['person_object_id', 'title']] \
        .merge(people_name, left_on='person_object_id', right_on='object_id', how='left') \
        .head(5)
    
    print(f"\n  Sample people records (first 5):")
    for _, row in rel_people_sample.iterrows():
        name = str(row.get('first_name', 'Unknown'))[:30] if pd.notna(row.get('first_name')) else 'Unknown'
        rel_title = str(row.get('title', 'N/A'))[:40] if pd.notna(row.get('title')) else 'N/A'
        print(f"    {name:30s} | {rel_title:40s}")
else:
    print("No relationships found.")

Top 20 titles in linked people:
     1. CEO                                                  720 ( 5.75%)
     2. Co-Founder                                           392 ( 3.13%)
     3. Board Member                                         384 ( 3.07%)
     4. CTO                                                  376 ( 3.00%)
     5. Founder                                              324 ( 2.59%)
     6. Advisor                                              299 ( 2.39%)
     7. Board of Directors                                   281 ( 2.24%)
     8. Director                                             227 ( 1.81%)
     9. Co-founder                                           180 ( 1.44%)
    10. CFO                                                  178 ( 1.42%)
    11. Investor                                             169 ( 1.35%)
    12. COO                                                  162 ( 1.29%)
    13. Founder & CEO                                        123 ( 0.98%)
    14

In [162]:
degrees.head()

,id,object_id,degree_type,subject,institution,graduated_at,created_at,updated_at
0,1,p:6117,MBA,NaN,NaN,NaN,2008-02-19 03:17:36,2008-02-19 03:17:36
1,2,p:6136,BA,"English, French","Washington University, St. Louis",1990-01-01,2008-02-19 17:58:31,2008-02-25 00:23:55
2,3,p:6136,MS,Mass Communication,Boston University,1992-01-01,2008-02-19 17:58:31,2008-02-25 00:23:55
3,4,p:6005,MS,Internet Technology,University of Greenwich,2006-01-01,2008-02-19 23:40:40,2008-02-25 00:23:55
4,5,p:5832,BCS,"Computer Science, Psychology",Rice University,NaN,2008-02-20 05:28:09,2008-02-20 05:28:09


In [163]:
print("="*80)
print("DEGREE COVERAGE [GATE-KEEPER METRIC]")
print("="*80)

# Match path: degrees.object_id -> relationships.person_object_id -> companies key via relationships.relationship_object_id
company_key = 'id' if 'id' in companies.columns else 'object_id'
valid_company_ids = set(companies[company_key].dropna().astype(str))

rel_for_companies = relationships[
    relationships['relationship_object_id'].astype(str).isin(valid_company_ids)
].copy()

matched_person_ids = set(rel_for_companies['person_object_id'].dropna().astype(str))
degrees_filtered = degrees[degrees['object_id'].astype(str).isin(matched_person_ids)].copy()

companies_with_degrees = rel_for_companies[
    rel_for_companies['person_object_id'].astype(str).isin(set(degrees_filtered['object_id'].astype(str)))
]['relationship_object_id'].nunique()
companies_with_degrees_pct = 100 * companies_with_degrees / len(companies)

print(f"\nCompanies with >= 1 linked person with degree record:")
print(f"{companies_with_degrees} / {len(companies)} ({companies_with_degrees_pct:.2f}%)")

# Print joined samples to verify mapping path end-to-end.
if len(rel_for_companies) > 0 and len(degrees_filtered) > 0:
    rel_tmp = rel_for_companies[['relationship_object_id', 'person_object_id', 'title']].copy()
    rel_tmp['company_id_str'] = rel_tmp['relationship_object_id'].astype(str)
    rel_tmp['person_id_str'] = rel_tmp['person_object_id'].astype(str)

    deg_tmp = degrees_filtered[['object_id', 'degree_type', 'institution']].copy()
    deg_tmp['person_id_str'] = deg_tmp['object_id'].astype(str)

    people_tmp = people[['object_id', 'first_name']].copy()
    people_tmp['person_id_str'] = people_tmp['object_id'].astype(str)

    comp_tmp = companies[[company_key, 'name']].copy()
    comp_tmp['company_id_str'] = comp_tmp[company_key].astype(str)

    sample_links = (
        rel_tmp
        .merge(people_tmp[['person_id_str', 'first_name']], on='person_id_str', how='left')
        .merge(deg_tmp[['person_id_str', 'degree_type', 'institution']], on='person_id_str', how='inner')
        .merge(comp_tmp[['company_id_str', 'name']], on='company_id_str', how='left')
        .head(5)
    )

    print(f"\nSample join checks (degrees.object_id -> relationships.person_object_id -> relationships.relationship_object_id -> companies.{company_key}):")
    for _, row in sample_links.iterrows():
        first_name = str(row.get('first_name', 'Unknown'))[:20] if pd.notna(row.get('first_name')) else 'Unknown'
        rel_title = str(row.get('title', 'N/A'))[:30] if pd.notna(row.get('title')) else 'N/A'
        degree_type = str(row.get('degree_type', 'N/A'))[:15] if pd.notna(row.get('degree_type')) else 'N/A'
        company_name = str(row.get('name', 'N/A'))[:35] if pd.notna(row.get('name')) else 'N/A'
        print(f"  Person {row['person_id_str'][:12]:12s} ({first_name:20s}) | Rel title: {rel_title:30s} | Degree: {degree_type:15s} | Company: {company_name}")
else:
    print("\nNo join samples available (missing relationship or degree matches).")


DEGREE COVERAGE [GATE-KEEPER METRIC]

Companies with >= 1 linked person with degree record:
1505 / 2653 (56.73%)

Sample join checks (degrees.object_id -> relationships.person_object_id -> relationships.relationship_object_id -> companies.object_id):
  Person p:30         (David               ) | Rel title: Founder & Chairman             | Degree: JD              | Company: Geni
  Person p:30         (David               ) | Rel title: Founder & Chairman             | Degree: BA              | Company: Geni
  Person p:31         (Alan                ) | Rel title: Board Member                   | Degree: BS              | Company: Geni
  Person p:35         (George              ) | Rel title: Board Member                   | Degree: N/A             | Company: Geni
  Person p:26         (Peter               ) | Rel title: Board of Directors             | Degree: BA              | Company: Geni


In [164]:
degree_types = degrees_filtered['degree_type'].value_counts()
print(f"\nDegree Type Distribution:")
for dtype, count in degree_types.items():
    pct = 100 * count / len(degrees_filtered)
    print(f"    {str(dtype)[:20]:20s} {count:6d} ({pct:5.2f}%)")

if len(degrees_filtered) > 0:
    print(f"\n  Sample degree records (first 5):")
    sample_deg = degrees_filtered[['object_id', 'degree_type', 'subject', 'institution']].head(5)
    for _, row in sample_deg.iterrows():
        subj = str(row['subject'])[:25] if pd.notna(row['subject']) else 'N/A'
        inst = str(row['institution'])[:30] if pd.notna(row['institution']) else 'N/A'
        print(f"    {row['degree_type']:10s} | {subj:25s} | {inst:30s}")


Degree Type Distribution:
    BS                     1900 (22.93%)
    BA                     1422 (17.16%)
    MBA                    1218 (14.70%)
    MS                      867 (10.46%)
    PhD                     275 ( 3.32%)
    Degree                  215 ( 2.60%)
    JD                      174 ( 2.10%)
    BBA                     121 ( 1.46%)
    BE                      110 ( 1.33%)
    MA                      105 ( 1.27%)
    BTECH                    52 ( 0.63%)
    MD                       34 ( 0.41%)
    BFA                      32 ( 0.39%)
    MENG                     27 ( 0.33%)
    BCOM                     24 ( 0.29%)
    Diploma                  23 ( 0.28%)
    LLB                      21 ( 0.25%)
    MSE                      20 ( 0.24%)
    MPHIL                    12 ( 0.14%)
    BENG                     11 ( 0.13%)
    Bachelor of Arts (B.     11 ( 0.13%)
    SB                       10 ( 0.12%)
    Master Degree             9 ( 0.11%)
    ME                        

In [165]:
# Binary feature: did this person attend a top university?
# Keep logic aligned with data_processing notebook by loading QS rankings dynamically.
TOP_UNIVERSITY_COUNT = 50
QS_RANKINGS_PATH = "../data/raw/2024 QS World University Rankings.csv"


def load_top_universities(rankings_path: str, top_n: int = 50) -> set[str]:
    rankings = pd.read_csv(rankings_path, low_memory=False)

    rank_cols = [col for col in rankings.columns if "RANK" in col.upper()]
    rank_col = next((col for col in rank_cols if "2024" in col.upper()), rank_cols[0] if rank_cols else None)
    if rank_col is None:
        raise ValueError(f"No rank column found in {rankings_path}")

    institution_col = next((col for col in rankings.columns if "INSTITUTION" in col.upper()), None)
    if institution_col is None:
        raise ValueError(f"No institution column found in {rankings_path}")

    top_rankings = rankings.copy()
    top_rankings[rank_col] = pd.to_numeric(
        top_rankings[rank_col].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce",
    )
    top_rankings = top_rankings.dropna(subset=[rank_col, institution_col])
    top_rankings = top_rankings.sort_values(rank_col)
    top_rankings = top_rankings[top_rankings[rank_col] <= top_n]

    return {
        str(name).strip().lower()
        for name in top_rankings[institution_col].dropna().tolist()
        if str(name).strip()
    }


TOP_UNIVERSITIES = load_top_universities(QS_RANKINGS_PATH, top_n=TOP_UNIVERSITY_COUNT)

degrees_filtered = degrees_filtered.copy()
degrees_filtered['institution_norm'] = degrees_filtered['institution'].fillna('').astype(str).str.lower()
degrees_filtered['top_university_flag'] = degrees_filtered['institution_norm'].apply(
    lambda inst: int(any(u in inst for u in TOP_UNIVERSITIES)) if inst else 0
)

# Optional person-level binary: 1 if any degree record is from a top university.
person_top_uni = (
    degrees_filtered.groupby('object_id', as_index=False)['top_university_flag']
    .max()
    .rename(columns={'top_university_flag': 'person_top_university_flag'})
)

print('Top University Flag (degree-row level):')
print(f"  Loaded top universities: {len(TOP_UNIVERSITIES)} from {QS_RANKINGS_PATH}")
print(f"  Flagged rows: {degrees_filtered['top_university_flag'].sum()} / {len(degrees_filtered)} ({100*degrees_filtered['top_university_flag'].mean():.2f}%)")
print(f"  Unique people with flag=1: {person_top_uni['person_top_university_flag'].sum()} / {len(person_top_uni)}")

# Print top attended institutions
flagged_institutions = degrees_filtered[degrees_filtered['top_university_flag'] == 1]['institution'].value_counts().head(10)
print(f"\nTop attended institutions:")
for i, (inst, count) in enumerate(flagged_institutions.items(), 1):
    pct = 100 * count / degrees_filtered['top_university_flag'].sum()
    print(f"  {i:2d}. {str(inst)[:50]:50s} {count:5d} ({pct:5.2f}%)")

print('\nSample flagged records:')
sample_top_uni = degrees_filtered[degrees_filtered['top_university_flag'] == 1][['object_id', 'degree_type', 'institution']].head(10)
for _, row in sample_top_uni.iterrows():
    degree_type = str(row.get('degree_type', 'N/A'))[:15] if pd.notna(row.get('degree_type')) else 'N/A'
    institution = str(row.get('institution', 'N/A'))[:50] if pd.notna(row.get('institution')) else 'N/A'
    print(f"  Person {str(row['object_id'])[:12]:12s} | {degree_type:15s} | {institution}")

Top University Flag (degree-row level):
  Loaded top universities: 50 from ../data/raw/2024 QS World University Rankings.csv
  Flagged rows: 2094 / 8285 (25.27%)
  Unique people with flag=1: 1606 / 5086

Top attended institutions:
   1. Stanford University                                  448 (21.39%)
   2. Harvard University                                   181 ( 8.64%)
   3. Massachusetts Institute of Technology (MIT)          132 ( 6.30%)
   4. Cornell University                                   124 ( 5.92%)
   5. Stanford University Graduate School of Business      107 ( 5.11%)
   6. University of Pennsylvania                            96 ( 4.58%)
   7. Columbia University                                   90 ( 4.30%)
   8. Princeton University                                  85 ( 4.06%)
   9. Yale University                                       80 ( 3.82%)
  10. University of Chicago                                 63 ( 3.01%)

Sample flagged records:
  Person p:2351       | 

In [166]:
# Build serial-entrepreneur stats from relationships, then fetch names from people and company names from companies.
company_key = 'id' if 'id' in companies.columns else 'object_id'

rel_serial = relationships_filtered[['person_object_id', 'relationship_object_id']].copy()
rel_serial['person_id_str'] = rel_serial['person_object_id'].astype(str)
rel_serial['company_id_str'] = rel_serial['relationship_object_id'].astype(str)

people_name_col = 'name' if 'name' in people.columns else 'first_name'
people_lookup = people[['object_id', people_name_col]].copy()
people_lookup['person_id_str'] = people_lookup['object_id'].astype(str)
people_lookup = people_lookup.rename(columns={people_name_col: 'person_name'})

company_lookup = companies[[company_key, 'name']].copy()
company_lookup['company_id_str'] = company_lookup[company_key].astype(str)
company_lookup = company_lookup.rename(columns={'name': 'company_name'})

companies_per_person = rel_serial.groupby('person_id_str')['company_id_str'].nunique()
serial_entrepreneurs = companies_per_person[companies_per_person >= 2].sort_values(ascending=False)

serial_count = len(serial_entrepreneurs)
total_people = rel_serial['person_id_str'].nunique()
serial_pct = 100 * serial_count / total_people if total_people > 0 else 0

print(f"Serial Entrepreneur Signal:")
print(f"  Total unique people: {total_people}")
print(f"  Linked to >= 2 companies: {serial_count} ({serial_pct:.2f}%)")

if serial_count > 0:
    print(f"\n  Top serial entrepreneurs (linked to most companies):")
    top_ids = serial_entrepreneurs.head(5).index.tolist()

    top_people = (
        pd.DataFrame({'person_id_str': top_ids})
        .merge(people_lookup[['person_id_str', 'person_name']], on='person_id_str', how='left')
    )

    for _, prow in top_people.iterrows():
        person_id = prow['person_id_str']
        person_name = str(prow.get('person_name', 'Unknown'))[:40] if pd.notna(prow.get('person_name')) else 'Unknown'
        company_count = int(serial_entrepreneurs.loc[person_id])

        linked_company_names = (
            rel_serial[rel_serial['person_id_str'] == person_id][['company_id_str']]
            .drop_duplicates()
            .merge(company_lookup[['company_id_str', 'company_name']], on='company_id_str', how='left')
            ['company_name']
            .dropna()
            .astype(str)
            .head(3)
            .tolist()
        )
        sample_companies = ', '.join(linked_company_names) if linked_company_names else 'N/A'
        print(f"    {person_name:40s} - {company_count} companies | Sample: {sample_companies}")

Serial Entrepreneur Signal:
  Total unique people: 11179
  Linked to >= 2 companies: 826 (7.39%)

  Top serial entrepreneurs (linked to most companies):
    Tim                                      - 12 companies | Sample: SnapTell, Dapper, Pelago
    Nihal                                    - 9 companies | Sample: MoVoxx, Greystripe, AdMob
    Pejman                                   - 8 companies | Sample: Powerset, TokBox, Social Gaming Network
    Saul                                     - 8 companies | Sample: SocialMedia.com, Kindo Network, Snaptalent
    Betty                                    - 7 companies | Sample: Grockit, Viki, Bitzer Mobile


## 5. Data Quality Checks

In [167]:
key_cols = ['object_id', 'name', 'overview', 'category_code', 'country_code',
            'funding_total_usd', 'founded_at', 'relationships', 'milestones']
print(f"Missing Values (key columns):")
for col in key_cols:
    if col in companies.columns:
        miss = companies[col].isna().sum()
        miss_pct = 100 * miss / len(companies)
        print(f"  {col:25s} {miss:5d} ({miss_pct:5.2f}%)")

Missing Values (key columns):
  object_id                     0 ( 0.00%)
  name                          0 ( 0.00%)
  overview                      0 ( 0.00%)
  category_code                52 ( 1.96%)
  country_code                335 (12.63%)
  funding_total_usd             0 ( 0.00%)
  founded_at                    0 ( 0.00%)
  relationships                 0 ( 0.00%)
  milestones                    0 ( 0.00%)


In [168]:
dup_ids = companies[companies.duplicated(subset=['object_id'], keep=False)]['object_id'].nunique()
dup_names = companies[companies.duplicated(subset=['name'], keep=False)]['name'].nunique()

print(f"Duplicate Entries:")
print(f"  Duplicate object_ids: {dup_ids}")
print(f"  Duplicate names: {dup_names}")

above_500m = (companies['funding_total_usd'] > 500_000_000).sum()
above_1b = (companies['funding_total_usd'] > 1_000_000_000).sum()

print(f"\nFunding Outliers:")
print(f"  Above USD 500M: {above_500m}")
print(f"  Above USD 1B: {above_1b}")

if above_500m > 0:
    print(f"\n  Notable high-funding companies:")
    high_funding = companies.nlargest(5, 'funding_total_usd')[['name', 'funding_total_usd', 'status']]
    for _, row in high_funding.iterrows():
        print(f"    {row['name'][:40]:40s} USD {row['funding_total_usd']:>15,.0f} ({row['status']})")

Duplicate Entries:
  Duplicate object_ids: 0
  Duplicate names: 0

Funding Outliers:
  Above USD 500M: 9
  Above USD 1B: 2

  Notable high-funding companies:
    Twitter                                  USD   1,160,166,511 (ipo)
    Groupon                                  USD   1,147,288,416 (ipo)
    Zynga                                    USD     860,213,000 (ipo)
    Better Place                             USD     835,750,000 (acquired)
    SolarCity                                USD     789,039,992 (ipo)


## Summary

In [169]:
print(f"Dataset: {len(companies)} companies (founded 2005-2013, known outcome)")
print(f"\nAgent Readiness:")

if cat_missing_pct < 15 and cat_nunique <= 25:
    print(f"  Market Analyst: READY")
else:
    print(f"  Market Analyst: NEEDS WORK")

if overview_stats['median'] >= 300 and len(companies_with_funding) > 0:
    print(f"  Business Model Analyst: READY")
else:
    print(f"  Business Model Analyst: NEEDS WORK")

if companies_with_milestones_pct >= 20 and companies_with_2plus_rounds_pct >= 30:
    print(f"  Feasibility Analyst: READY")
else:
    print(f"  Feasibility Analyst: NEEDS WORK")

if companies_with_degrees_pct >= 40:
    print(f"  Team Analyst: STRONG (proceed as planned)")
elif companies_with_degrees_pct >= 25:
    print(f"  Team Analyst: WEAK (consider modifications)")
else:
    print(f"  Team Analyst: TOO WEAK (reconsider design)")

Dataset: 2653 companies (founded 2005-2013, known outcome)

Agent Readiness:
  Market Analyst: NEEDS WORK
  Business Model Analyst: READY
  Feasibility Analyst: NEEDS WORK
  Team Analyst: STRONG (proceed as planned)
